In [ ]:
import pandas as pd
import json
import os
import http.server
import socketserver
import webbrowser

admin_df = pd.read_csv('../data/processed/행정동.csv')
# ==============================================================================
# 1. 설정 및 데이터 준비
# ==============================================================================

# 카카오맵 JavaScript 키 (⚠️ 반드시 본인의 키로 교체하세요!)
JAVASCRIPT_KEY = "87119a9a672d25a5916aa93907429177" 
HTML_FILE_NAME = "store_locations_map.html"
PORT = 8000 # 로컬 서버 포트

# 4000개의 예시 데이터 생성 (실제 DataFrame을 사용하세요)
df = admin_df.copy()

# 지도를 위한 데이터 구조를 JavaScript JSON 문자열로 변환
store_list = df[['가맹점구분번호', '위도(lat)', '경도(lon)']].to_dict('records')
store_data_json = json.dumps(store_list, ensure_ascii=False)


# ==============================================================================
# 2. HTML 파일 생성 함수
# ==============================================================================

def generate_bulk_map_html(store_data_json, api_key, num_stores):
    """JSON 데이터를 받아 여러 마커를 표시하는 HTML 코드를 생성합니다."""
    
    html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>총 {df.shape[0]}개 가맹점 위치</title>
</head>
<body>
    <h2 style="text-align:center;">총 {df.shape[0]}개 가맹점 위치</h2>
    <div id="map" style="width:90%;height:800px;margin: 20px auto; border: 1px solid #ccc;"></div>
    
    <script type="text/javascript" src="//dapi.kakao.com/v2/maps/sdk.js?appkey={api_key}"></script>
    <script>
        // Python에서 전달된 가맹점 데이터
        var storeData = {store_data_json};

        // 지도 컨테이너 및 옵션 설정 (데이터의 평균 좌표를 중심으로 설정)
        // 실제 마커를 표시할 때 성능 저하를 줄이기 위해 클러스터링을 고려해야 합니다.
        var mapContainer = document.getElementById('map'); 
        
        // 중심 좌표를 전체 데이터의 대략적인 중앙으로 설정 (간단하게 첫 번째 데이터 사용)
        var centerLat = storeData.reduce((sum, d) => sum + d.latitude, 0) / storeData.length;
        var centerLng = storeData.reduce((sum, d) => sum + d.longitude, 0) / storeData.length;

        var mapOption = {{ 
            center: new kakao.maps.LatLng(centerLat, centerLng),
            level: 7 
        }};

        var map = new kakao.maps.Map(mapContainer, mapOption); 
        
        // 인포윈도우 관리용 변수
        var currentInfowindow = null; 

        // 마커 및 인포윈도우 생성
        for (var i = 0; i < storeData.length; i++) {{
            var store = storeData[i];
            
            var markerPosition  = new kakao.maps.LatLng(store.latitude, store.longitude); 
            
            var marker = new kakao.maps.Marker({{
                position: markerPosition,
                title: store.store_name 
            }});
            
            marker.setMap(map);

            var infowindow = new kakao.maps.InfoWindow({{
                content : '<div style="padding:5px;font-size:12px;">' + store.store_name + '</div>'
            }});

            // 마커 클릭 시 인포윈도우 표시 이벤트 추가
            kakao.maps.event.addListener(marker, 'click', (function(map, marker, infowindow) {{
                return function() {{
                    if (currentInfowindow) {{
                        currentInfowindow.close();
                    }}
                    infowindow.open(map, marker);
                    currentInfowindow = infowindow;
                }};
            }})(map, marker, infowindow));
        }}
        
        // 지도 클릭 시 열려있는 인포윈도우 닫기
        kakao.maps.event.addListener(map, 'click', function() {{
            if (currentInfowindow) {{
                currentInfowindow.close();
                currentInfowindow = null;
            }}
        }});
    </script>
</body>
</html>
"""
    return html_template

# HTML 파일 생성 및 저장
html_output = generate_bulk_map_html(store_data_json, JAVASCRIPT_KEY, df.shape[0])

with open(HTML_FILE_NAME, "w", encoding="utf-8") as f:
    f.write(html_output)

print(f"✅ 지도 HTML 파일 생성 완료: {os.path.abspath(HTML_FILE_NAME)}")


# ==============================================================================
# 3. 로컬 서버 실행 및 브라우저 열기
# ==============================================================================

Handler = http.server.SimpleHTTPRequestHandler

try:
    with socketserver.TCPServer(("", PORT), Handler) as httpd:
        server_url = f"http://127.0.0.1:{PORT}/{HTML_FILE_NAME}"
        
        print("-" * 50)
        print("💡 로컬 서버가 시작되었습니다.")
        print(f"   카카오맵에 등록된 도메인이 'http://127.0.0.1'인지 확인하세요.")
        print(f"   브라우저에서 다음 주소로 접속하세요: {server_url}")
        print("-" * 50)
        
        # 브라우저 자동 실행
        webbrowser.open_new_tab(server_url)
        
        # 서버 유지
        httpd.serve_forever()

except Exception as e:
    print(f"\n❌ 서버 실행 중 오류 발생. (포트 {PORT} 사용 중일 수 있음): {e}")